# Stage 7 — Prompt template + LLM generation

Input: `data/eval_set.jsonl` (20 questions with known-correct chunks) + `data/chunks/chunks.jsonl`.

Retrieval method carried forward from Stage 6: **hybrid** (vector top-15 + BM25 top-15, fused via RRF to a final top-5) — it scored the best Recall@5 (0.90) of the three methods compared, and recall matters more than diversity here since the LLM can't answer correctly from context that simply doesn't contain the answer.

This notebook:

1. Reuses the hybrid retrieval pipeline from Stage 6 unchanged
2. Builds a prompt template that separates text/table/image context clearly and enforces grounding (answer only from the given sources)
3. Generates a structured JSON answer (`answer` + `sources`) per question via GPT-4o
4. Logs cost/latency the same way Stage 3 did
5. Saves everything to `data/generations.jsonl` — the direct input for Stage 8 (RAGAS) and Stage 9 (DOCX export)

Requires Milvus running and the same Azure OpenAI `.env` vars as Stage 6.


In [1]:
import sys
import importlib.metadata as md

print("Python:", sys.executable)

for pkg in [
    "ragas",
    "openai",
    "langchain-community",
    "langchain-openai",
]:
    try:
        print(pkg, md.version(pkg))
    except md.PackageNotFoundError:
        print(pkg, "NOT INSTALLED")

Python: /Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag/.venv/bin/python
ragas 0.3.1
openai 3.8.0
langchain-community 0.4.2
langchain-openai 1.6.0


In [1]:
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import AzureOpenAI
from pymilvus import connections, Collection
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

/Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CHUNKS_PATH = DATA_DIR / "chunks" / "chunks.jsonl"
EVAL_SET_PATH = DATA_DIR / "eval_set.jsonl"
LOGS_DIR = DATA_DIR / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
AZURE_OPENAI_GPT4O_DEPLOYMENT = os.environ["AZURE_OPENAI_GPT4O_DEPLOYMENT"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

connections.connect(alias="default", host="localhost", port="19530")

CHOSEN_INDEX = "hnsw"
SEARCH_CONFIG = {"metric_type": "COSINE", "search_params": {"ef": 64}}
CANDIDATE_POOL_SIZE = 15
FINAL_K = 5
RRF_K = 60

usage_log: list[dict] = []


def record_usage(call_type: str, n_items: int, latency_sec: float, prompt_tokens: int = 0, completion_tokens: int = 0) -> None:
    usage_log.append({
        "call_type": call_type,
        "n_items": n_items,
        "latency_sec": latency_sec,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": prompt_tokens + completion_tokens,
    })


with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = [json.loads(line) for line in f]
chunks_by_id = {c["chunk_id"]: c for c in all_chunks}

with open(EVAL_SET_PATH, "r", encoding="utf-8") as f:
    eval_set = [json.loads(line) for line in f]

print(f"loaded {len(all_chunks)} chunks, {len(eval_set)} eval questions")

loaded 1301 chunks, 20 eval questions


/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_45451/1388256017.py:22: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(alias="default", host="localhost", port="19530")


In [3]:
collection = Collection(f"chunks_{CHOSEN_INDEX}")
collection.load()


def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


bm25_corpus_ids = [c["chunk_id"] for c in all_chunks]
bm25 = BM25Okapi([tokenize(c["content"]) for c in all_chunks])

print(f"chunks_{CHOSEN_INDEX} ready ({collection.num_entities} rows), BM25 index built")

chunks_hnsw ready (1301 rows), BM25 index built


/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_45451/3614854069.py:1: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection(f"chunks_{CHOSEN_INDEX}")
/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_45451/3614854069.py:2: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.load()
/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_45451/3614854069.py:12: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print(f"chunks_{CHOSEN_INDEX} ready ({collection.num_entities} rows), BM25 index built")


## Retrieval pipeline (unchanged from Stage 6)

`get_hybrid_context` is the single entry point: embed the query, get vector + BM25 candidate pools, fuse via RRF, return the final top-5 chunks — exactly the winning method from Stage 6's comparison.


In [4]:
def embed_query(text: str) -> list[float]:
    response = client.embeddings.create(model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT, input=text)
    return response.data[0].embedding


def get_vector_candidates(query_text: str, top_k: int = CANDIDATE_POOL_SIZE) -> list[dict]:
    query_vector = embed_query(query_text)
    results = collection.search(
        data=[query_vector],
        anns_field="embedding",
        param=SEARCH_CONFIG,
        limit=top_k,
        output_fields=["chunk_id"],
    )[0]
    return [{"chunk_id": hit.entity.get("chunk_id")} for hit in results]


def get_bm25_candidates(query_text: str, top_k: int = CANDIDATE_POOL_SIZE) -> list[str]:
    scores = bm25.get_scores(tokenize(query_text))
    ranked_idx = np.argsort(scores)[::-1][:top_k]
    return [bm25_corpus_ids[i] for i in ranked_idx]


def rrf_fuse(vector_candidates: list[dict], bm25_chunk_ids: list[str], final_k: int = FINAL_K) -> list[str]:
    rrf_scores = {}

    for rank, c in enumerate(vector_candidates, start=1):
        rrf_scores[c["chunk_id"]] = rrf_scores.get(c["chunk_id"], 0) + 1 / (RRF_K + rank)

    for rank, chunk_id in enumerate(bm25_chunk_ids, start=1):
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (RRF_K + rank)

    return sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:final_k]


def get_hybrid_context(query_text: str, k: int = FINAL_K) -> list[dict]:
    vector_candidates = get_vector_candidates(query_text)
    bm25_chunk_ids = get_bm25_candidates(query_text)
    top_ids = rrf_fuse(vector_candidates, bm25_chunk_ids, final_k=k)
    return [chunks_by_id[cid] for cid in top_ids]

## Prompt template

`build_context_block` labels each source with its type and location (`[Source N — table, doc p.X]`) so the model knows what kind of evidence it's reading, rather than an undifferentiated wall of text.

The prompt itself enforces grounding — answer only from the given sources, say so explicitly if they're insufficient — which matters directly for Stage 8's faithfulness scoring.

The model is asked to return **JSON** (`answer` + `sources`) via GPT-4o's JSON-mode response format, which is easier to parse reliably than extracting citations from free-form prose, and is exactly the shape Stage 9's DOCX export needs.


In [5]:
def build_context_block(context_chunks: list[dict]) -> str:
    blocks = []
    for i, chunk in enumerate(context_chunks, start=1):
        header = f"[Source {i} — {chunk['chunk_type']}, {chunk['doc_id']} p.{chunk['page_number']}]"
        if chunk["chunk_type"] == "table":
            blocks.append(f"{header}\nTable:\n{chunk['content']}")
        elif chunk["chunk_type"] == "image":
            blocks.append(f"{header}\nFigure description: {chunk['content']}")
        else:
            blocks.append(f"{header}\n{chunk['content']}")
    return "\n\n".join(blocks)


GENERATION_PROMPT = """You are answering a question using ONLY the numbered sources below. Do not use any outside knowledge. If the sources do not contain enough information to answer, say so explicitly rather than guessing.

{context}

Question: {question}

Respond with a JSON object in exactly this shape:
{{
  "answer": "<your answer, citing sources inline like [Source 1] where relevant>",
  "sources": [
    {{"source_number": <int>, "doc_id": "<string>", "page_number": <int>, "chunk_type": "<text|table|image>"}}
  ]
}}
Only include sources in the "sources" list that you actually used to answer."""

## Generation

Times and logs the call like Stage 3's captioning did.

If the model's JSON somehow fails to parse (rare with JSON mode, but not impossible), it falls back to treating the raw text as the answer with an empty source list rather than crashing the whole batch on one bad response.


In [6]:
def generate_answer(question: str, context_chunks: list[dict]) -> dict:
    prompt = GENERATION_PROMPT.format(context=build_context_block(context_chunks), question=question)

    start = time.perf_counter()
    response = client.chat.completions.create(
        model=AZURE_OPENAI_GPT4O_DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        max_tokens=600,
    )
    latency = time.perf_counter() - start

    record_usage(
        "generation",
        n_items=1,
        latency_sec=latency,
        prompt_tokens=response.usage.prompt_tokens,
        completion_tokens=response.usage.completion_tokens,
    )

    content = response.choices[0].message.content
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {"answer": content, "sources": []}

#### Sanity check on one question before running all 20"

In [7]:
sample_item = eval_set[0]
sample_context = get_hybrid_context(sample_item["question"])
sample_result = generate_answer(sample_item["question"], sample_context)

print("Q:", sample_item["question"])
print("A:", sample_result.get("answer"))
print("Sources:", sample_result.get("sources"))

/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_45451/3537724094.py:8: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


Q: What demonstrates more variation in estimates of ECS, the upper bounds or the lower bounds?
A: Estimates of ECS demonstrate more variation in the upper bounds than in the lower bounds [Source 1, Source 3].
Sources: [{'source_number': 1, 'doc_id': 'IPCC_AR6_WGI_Chapter07.pdf', 'page_number': 80, 'chunk_type': 'text'}, {'source_number': 3, 'doc_id': 'IPCC_AR6_WGI_Chapter07.pdf', 'page_number': 84, 'chunk_type': 'text'}]


#### Generate an answer for all 20 questions"

In [8]:
generation_records = []

for item in tqdm(eval_set, desc="generating answers"):
    context_chunks = get_hybrid_context(item["question"])
    result = generate_answer(item["question"], context_chunks)

    generation_records.append({
        "question": item["question"],
        "expected_chunk_id": item["expected_chunk_id"],
        "context_chunk_ids": [c["chunk_id"] for c in context_chunks],
        "answer": result.get("answer", ""),
        "sources": result.get("sources", []),
    })

generations_df = pd.DataFrame(generation_records)
generations_df.head(10)

generating answers:   0%|          | 0/20 [00:00<?, ?it/s]/var/folders/9p/gm2n68xn7n1bhds792y57yg40000gn/T/ipykernel_45451/3537724094.py:8: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(
generating answers: 100%|██████████| 20/20 [00:40<00:00,  2.05s/it]


,question,expected_chunk_id,context_chunk_ids,answer,sources
0,What demonstrates more variation in estimates ...,271c6890dc2a430ebf22ae950ab009f3,"[271c6890dc2a430ebf22ae950ab009f3, 6ee29b82fdb...",The upper bounds of ECS estimates demonstrate ...,"[{'source_number': 1, 'doc_id': 'IPCC_AR6_WGI_..."
1,What advantage does the radiative kernel appro...,6d8d07b79d8b406cbd564d578ccd3ccb,"[6d8d07b79d8b406cbd564d578ccd3ccb, 597e95f19e3...",The radiative kernel approach is advantageous ...,"[{'source_number': 1, 'doc_id': 'IPCC_AR6_WGI_..."
2,What factors contribute to permafrost degradat...,028ca87fa9374bab8127d7ec83030a96,"[028ca87fa9374bab8127d7ec83030a96, 930b2329e47...",Factors contributing to permafrost degradation...,"[{'source_number': 1, 'doc_id': 'IPCC_AR6_WGI_..."
3,What is the title of the study published by Po...,dc592958e93b4406881c295369e45fc6,"[dc592958e93b4406881c295369e45fc6, 324c6d88cec...",The title of the study published by Ponater in...,"[{'source_number': 1, 'doc_id': 'IPCC_AR6_WGI_..."
4,What is the title and publication year of the ...,3b9e59df2c784b75a8b7e27150ce6a81,"[1952d5670fec49eba5f5c19813f9488d, 6738c3077e7...",The title of the study is 'Observational Evide...,"[{'source_number': 4, 'doc_id': 'IPCC_AR6_WGI_..."
5,How do aerosols released by human activities i...,dbeb7523ea8744e3918d230b0f0c10e5,"[dbeb7523ea8744e3918d230b0f0c10e5, 81d02f76164...",Aerosols released by human activities influenc...,"[{'source_number': 1, 'doc_id': 'IPCC_AR6_WGI_..."
6,What evidence suggests that the true equilibri...,eafa938cb2b846b29ec4e5d59278aad1,"[07312213bc94444abde5adfbceefccf5, 2e678b29224...",The evidence suggesting that the true equilibr...,"[{'source_number': 5, 'doc_id': 'IPCC_AR6_WGI_..."
7,What is the journal title and volume number in...,f0480d0c42764a76b48dbc1eaef9c5a2,"[f0480d0c42764a76b48dbc1eaef9c5a2, 05cf4e73f2f...",The study by Diffenbaugh and Singh in 2014 on ...,"[{'source_number': 1, 'doc_id': 'IPCC_AR6_WGI_..."
8,What effect could freshwater input from meltin...,0b2478fcfba34aad945e25679cdb5c90,"[0b2478fcfba34aad945e25679cdb5c90, ec11f533d92...",Freshwater input from melting ice shelves coul...,"[{'source_number': 1, 'doc_id': 'IPCC_AR6_WGI_..."
9,What study analyzed the amplification of the N...,e412caa306914cabba9a44e0b7ca96eb,"[e412caa306914cabba9a44e0b7ca96eb, 9d10fcd26ec...","The study by Cook, B.I., R.L. Miller, and R. S...","[{'source_number': 1, 'doc_id': 'IPCC_AR6_WGI_..."


## Save generations + usage summary

`data/generations.jsonl` is the direct input for Stage 8 (RAGAS needs question + context + answer) and Stage 9 (DOCX export needs answer + sources).

The usage summary follows the same pattern as Stage 3 — real token counts from `response.usage`, with a placeholder price table to verify against current rates.


In [9]:
def save_jsonl(units: list[dict], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for unit in units:
            f.write(json.dumps(unit) + "\n")


save_jsonl(generation_records, DATA_DIR / "generations.jsonl")
save_jsonl(usage_log, LOGS_DIR / "stage7_usage_log.jsonl")

PRICE_PER_1K_TOKENS = {
    "prompt": 0.0025,      # gpt-4o input -- VERIFY against current pricing
    "completion": 0.01,    # gpt-4o output -- VERIFY against current pricing
}

usage_df = pd.DataFrame(usage_log)
usage_df["estimated_cost_usd"] = (
    usage_df["prompt_tokens"] / 1000 * PRICE_PER_1K_TOKENS["prompt"]
    + usage_df["completion_tokens"] / 1000 * PRICE_PER_1K_TOKENS["completion"]
)

summary = pd.Series({
    "calls": len(usage_df),
    "total_tokens": usage_df["total_tokens"].sum(),
    "mean_latency_sec": usage_df["latency_sec"].mean(),
    "p95_latency_sec": np.percentile(usage_df["latency_sec"], 95),
    "total_estimated_cost_usd": usage_df["estimated_cost_usd"].sum(),
})

print(f"saved {len(generation_records)} generations to {DATA_DIR / 'generations.jsonl'}")
display(summary)

saved 20 generations to /Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag/data/generations.jsonl


calls                          21.000000
total_tokens                71968.000000
mean_latency_sec                1.837996
p95_latency_sec                 2.583336
total_estimated_cost_usd        0.198940
dtype: float64

## Next: Stage 8 — RAGAS evaluation

Each record in `data/generations.jsonl` has exactly what RAGAS needs: `question`, the retrieved `context_chunk_ids` (resolved back to content via `chunks_by_id`), and the generated `answer`.

Stage 8 scores:

* **Faithfulness**
* **Answer relevance**
* **Context precision**
* **Context recall**

across these 20 generated answers.
